# 1. Inspect the model

Builds the transformer autoencoder exactly as in the architecture diagram (15-layer encoder, 15-layer decoder, `d_model=1536`, 24 heads, 32k vocab, tied embeddings) and confirms the parameter count and forward-pass shapes.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import torch
from model.config import ModelConfig
from model.autoencoder import TransformerAutoencoder

In [ ]:
cfg = ModelConfig()
model = TransformerAutoencoder(cfg)

print(f"d_model={cfg.d_model}  n_heads={cfg.n_heads}  "
      f"enc_layers={cfg.n_encoder_layers}  dec_layers={cfg.n_decoder_layers}  "
      f"d_ff={cfg.d_ff}  vocab={cfg.vocab_size}  tied={cfg.tie_embeddings}")
n = model.num_parameters()
print(f"total parameters: {n:,}  (~{n / 1e9:.3f}B)")

Real forward pass on random token ids, just to confirm every shape lines up end to end (embeddings -> encoder -> latent Z -> decoder -> vocab projection).

In [ ]:
b, t = 2, 32
noisy = torch.randint(5, cfg.vocab_size, (b, t))
decoder_input = torch.randint(5, cfg.vocab_size, (b, t))

with torch.no_grad():
    logits = model(noisy, decoder_input)

print("logits shape:", tuple(logits.shape))
assert logits.shape == (b, t, cfg.vocab_size)
print("OK")

**Scope note:** this confirms the architecture is correctly wired at ~1.04B parameters. Actually pretraining it to convergence needs a many-billion-token corpus and multi-GPU compute -- see `04_train_autoencoder.ipynb` and the README for the demo-scale training loop this repo ships with.